<a href="https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Subhash-2910/flyrank-ML-T1.git"
REPO_DIR = "flyrank-ML-T1"

def find_repo_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data/raw/content_refresh_anonymized.csv").exists():
            return folder
    return None

repo_root = find_repo_root()

if IN_COLAB and repo_root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    repo_root = Path(REPO_DIR)

os.chdir(repo_root)

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Working directory:", os.getcwd())
print("Dataset shape:", df.shape)
display(df.head(3))

Working directory: /content/flyrank-ML-T1
Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I built a conservative feature vector for content-review scoring. It uses descriptive content, competition, age, and freshness fields that are available at the review snapshot. I excluded IDs, trend labels, and historical performance-window fields from the feature vector.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
]

feature_columns = numeric_features + categorical_features

X_raw = df[feature_columns].copy()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X = preprocessor.fit_transform(X_raw)

feature_names = preprocessor.get_feature_names_out()

print("Raw feature columns:", feature_columns)
print("Feature matrix shape:", X.shape)
print("Number of encoded features:", len(feature_names))
print("\nFirst encoded feature names:")
print(feature_names[:20])

Raw feature columns: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'competition_level', 'content_type', 'main_intent']
Feature matrix shape: (30000, 17)
Number of encoded features: 17

First encoded feature names:
['numeric__search_volume' 'numeric__competition' 'numeric__cpc'
 'numeric__word_count' 'numeric__char_count' 'numeric__content_age_days'
 'numeric__days_since_last_update' 'categorical__competition_level_HIGH'
 'categorical__competition_level_LOW'
 'categorical__competition_level_MEDIUM'
 'categorical__content_type_comparison article'
 'categorical__content_type_feedly article'
 'categorical__content_type_keyword article'
 'categorical__main_intent_commercial'
 'categorical__main_intent_informational'
 'categorical__main_intent_navigational'
 'categorical__main_intent_transactional']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Numeric fields use median filling so missing values do not remove a page from review. Categorical fields use the most common available category and are one-hot encoded. Search demand, competition, content length, content age, and days since update are descriptive snapshot fields. They can support prioritisation, but they do not prove that refreshing a page will change its ranking or traffic.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_notes = pd.DataFrame({
    "feature": feature_columns,
    "type": ["numeric"] * len(numeric_features) + ["categorical"] * len(categorical_features),
    "missing_values": [df[col].isna().sum() for col in feature_columns],
    "available_at_review_time": ["Yes"] * len(feature_columns),
    "handling": (
        ["Median imputation"] * len(numeric_features)
        + ["Most-frequent fill + one-hot encoding"] * len(categorical_features)
    )
})

display(feature_notes)

,feature,type,missing_values,available_at_review_time,handling
0,search_volume,numeric,2468,Yes,Median imputation
1,competition,numeric,2468,Yes,Median imputation
2,cpc,numeric,2468,Yes,Median imputation
3,word_count,numeric,7699,Yes,Median imputation
4,char_count,numeric,7699,Yes,Median imputation
5,content_age_days,numeric,0,Yes,Median imputation
6,days_since_last_update,numeric,0,Yes,Median imputation
7,competition_level,categorical,2610,Yes,Most-frequent fill + one-hot encoding
8,content_type,categorical,0,Yes,Most-frequent fill + one-hot encoding
9,main_intent,categorical,2374,Yes,Most-frequent fill + one-hot encoding


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the final feature vector for direct identifiers, label-derived fields, future or comparison-window performance fields, and product-style outcome flags. No unsafe field is included in the feature vector.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unsafe_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

leakage_check = pd.DataFrame({
    "column": unsafe_columns,
    "present_in_dataset": [column in df.columns for column in unsafe_columns],
    "used_as_feature": [column in feature_columns for column in unsafe_columns],
    "reason": [
        "Pseudonymous content identifier",
        "Pseudonymous client identifier; grouping only",
        "Target/label-derived decline direction",
        "Target/label-derived movement value",
        "Historical performance total",
        "Historical performance total",
        "Historical performance total",
        "Historical performance total",
        "Historical performance total",
        "Historical performance total",
        "Historical performance total",
        "Historical performance total",
        "Comparison-window performance field",
        "Comparison-window performance field",
        "Comparison-window performance field",
        "Comparison-window performance field",
        "Comparison-window performance field",
        "Comparison-window performance field",
    ]
})

display(leakage_check)

assert not leakage_check["used_as_feature"].any(), (
    "Unsafe fields were found in the feature vector."
)

print("PASS: No IDs, trend labels, or performance-window fields are used as features.")

,column,present_in_dataset,used_as_feature,reason
0,content_id,True,False,Pseudonymous content identifier
1,client_id,True,False,Pseudonymous client identifier; grouping only
2,trend_direction,True,False,Target/label-derived decline direction
3,trend_pct,True,False,Target/label-derived movement value
4,impressions_90d,True,False,Historical performance total
5,clicks_90d,True,False,Historical performance total
6,pageviews_90d,True,False,Historical performance total
7,sessions_90d,True,False,Historical performance total
8,users_90d,True,False,Historical performance total
9,engaged_sessions_90d,True,False,Historical performance total


PASS: No IDs, trend labels, or performance-window fields are used as features.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I excluded content_id and client_id because they are identifiers, not generalisable content signals. I excluded trend_direction and trend_pct because they define the movement outcome and would leak the answer. I also excluded the 90-day totals and current-versus-previous 30-day fields because they are performance-window fields that can overlap with, or make a decline target too easy to reconstruct. These exclusions make the feature vector more conservative and more appropriate for decision-support.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_summary = leakage_check[[
    "column",
    "reason"
]].copy()

display(excluded_summary)
print("Excluded field count:", len(excluded_summary))

,column,reason
0,content_id,Pseudonymous content identifier
1,client_id,Pseudonymous client identifier; grouping only
2,trend_direction,Target/label-derived decline direction
3,trend_pct,Target/label-derived movement value
4,impressions_90d,Historical performance total
5,clicks_90d,Historical performance total
6,pageviews_90d,Historical performance total
7,sessions_90d,Historical performance total
8,users_90d,Historical performance total
9,engaged_sessions_90d,Historical performance total


Excluded field count: 18


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.